<a href="https://colab.research.google.com/github/shivamjislt97/Mega-To-Gdrive-Transfer-limit-Bypass-new-trick-by-Shivam-SLT/blob/main/Mega_To_Gdrive_Transfer_limit_Bypass_new_trick_by_Shivam_SLT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Intro**

In [ ]:
import os
if os.path.exists("/content/mega_transfer_state.json"):
    os.remove("/content/mega_transfer_state.json")
    print("Old state file removed — starting fresh")

Old state file removed — starting fresh


# MEGA → Google Drive Transfer (No Telegram)

Individual MEGA public file links ko seedha Google Drive me copy karta hai.
Koi Telegram bot nahi — sab kuch is notebook ke output me dikhega.

**Resume:** Session disconnect ho jaaye to bas Cells 2 → 6 phir se run karo.
Yeh Drive folder ka real scan + state.json dono check karke sirf missing
files download karega, kuch duplicate nahi hoga.

**Limitation:** Colab free tier sessions kabhi bhi timeout ho sakte hain —
isse resume automatically handle karta hai.

**Cell 2: Install Dependencies**

In [ ]:
%%bash
set -e
echo "=== Setup ==="
if ! grep -q "universe" /etc/apt/sources.list 2>/dev/null; then
    sudo add-apt-repository universe -y || echo "WARN: could not add universe repo"
fi
apt-get update -qq || echo "WARN: apt-get update had issues"

echo "=== Installing megatools ==="
apt-get install -y megatools -qq

echo "=== Verification ==="
if command -v megadl >/dev/null 2>&1; then
    echo "megatools (megadl) installed successfully"
    megadl --version 2>&1 | head -1
else
    echo "FAILED: megadl not found after install — check errors above"
fi

echo "=== Installing mega.py (metadata-only pre-check, no download) ==="
pip install mega.py --break-system-packages -q
pip install --upgrade "tenacity>=8.2.0" --break-system-packages -q
python3 -c "from mega import Mega; print('mega.py import OK')"

**Cell 3: Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

**Cell 4: Configuration**

In [ ]:
import os, json, re

# ====== FILL THIS IN ======
MEGA_FILE_LINKS = [
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
    "https://mega.nz/file/.............................................",
]
DRIVE_DEST_FOLDER_NAME = "MEGA_Transfer"
# ===========================

DRIVE_DEST_PATH = f"/content/drive/MyDrive/{DRIVE_DEST_FOLDER_NAME}"
TEMP_DIR = "/content/mega_temp"
STATE_FILE = "/content/mega_transfer_state.json"
MAX_RETRIES = 3
BACKOFF_BASE_SECONDS = 5

os.makedirs(DRIVE_DEST_PATH, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

assert len(MEGA_FILE_LINKS) > 0 and all("mega.nz/file/" in l for l in MEGA_FILE_LINKS), \
    "Put at least one valid MEGA file link (mega.nz/file/...) in MEGA_FILE_LINKS."

def link_id(url):
    m = re.search(r"/file/([^#]+)", url)
    return m.group(1) if m else url

print(f"Config OK — {len(MEGA_FILE_LINKS)} file link(s) queued")
print("Drive destination:", DRIVE_DEST_PATH)
print("Temp dir:", TEMP_DIR)

**Cell 5: State Management + Drive Scan**

In [ ]:
import os, json

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, OSError) as e:
            print(f"WARN: could not read state file ({e}), starting fresh")
    return {}

def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

def scan_drive_folder():
    """Real Drive folder contents — source of truth. {filename: size_bytes}"""
    existing = {}
    for root, _, filenames in os.walk(DRIVE_DEST_PATH):
        for fname in filenames:
            full = os.path.join(root, fname)
            try:
                existing[fname] = os.path.getsize(full)
            except OSError:
                continue
    return existing

state = load_state()
drive_files = scan_drive_folder()
print(f"state.json: {len(state)} tracked link(s) from previous runs")
print(f"Drive folder scan (source of truth): {len(drive_files)} file(s) present")
if drive_files:
    for fname, size in list(drive_files.items())[:10]:
        print(f"  {fname} ({size} bytes)")

**Cell 6: Sequential Download Loop**

In [ ]:
import time, shutil, subprocess, os, sys, re
from tqdm.notebook import tqdm
from mega import Mega

QUOTA_ERROR_MARKERS = ["over quota", "bandwidth limit", "quota exceeded", "429", "eoverquota"]

def is_quota_error(text):
    return any(marker in text.lower() for marker in QUOTA_ERROR_MARKERS)

def _run(cmd, timeout=None):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout, p.stderr

def already_done(url, drive_files, state):
    """Fast check via state.json (no network call)."""
    rec = state.get(link_id(url))
    if not rec or rec.get("status") != "completed":
        return False
    fname, size = rec.get("filename"), rec.get("size")
    return fname is not None and drive_files.get(fname) == size

def get_remote_metadata(url):
    """Fetch (filename, size) from MEGA WITHOUT downloading content."""
    m = Mega()
    info = m.get_public_url_info(url)   # {'name': ..., 'size': ...}
    return info["name"], info["size"]

def already_done_by_metadata(url, drive_files):
    """Pre-check via real MEGA metadata before downloading."""
    try:
        fname, size = get_remote_metadata(url)
    except Exception as e:
        print(f"  WARN: could not fetch metadata ({e}), will download to check.")
        return False, None, None
    return (drive_files.get(fname) == size), fname, size

def clear_temp_dir():
    for f in os.listdir(TEMP_DIR):
        fp = os.path.join(TEMP_DIR, f)
        if os.path.isfile(fp):
            os.remove(fp)

def download_one(url, total_size=None, pbar=None):
    """Download with a REAL byte-based progress bar (actual visual fill)."""
    clear_temp_dir()
    cmd = ["megadl", "--path", TEMP_DIR, url]
    print(f"  Running: {' '.join(cmd)}")

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0
    )

    buf = b""
    last_line = ""
    bytes_re = re.compile(r"\(([\d,]+)\s*bytes\)")

    inner = tqdm(total=total_size, unit="B", unit_scale=True, unit_divisor=1024,
                 desc=f"  {link_id(url)}", leave=False)

    while True:
        chunk = process.stdout.read(1)
        if not chunk:
            break
        if chunk in (b"\r", b"\n"):
            if buf:
                line = buf.decode(errors="replace").strip()
                if line:
                    last_line = line
                    m = bytes_re.search(line)
                    if m:
                        downloaded = int(m.group(1).replace(",", ""))
                        inner.n = min(downloaded, total_size) if total_size else downloaded
                        inner.refresh()
                        if pbar is not None and total_size:
                            pct = downloaded / total_size * 100
                            pbar.set_postfix_str(f"{link_id(url)} {pct:.1f}%")
                buf = b""
        else:
            buf += chunk

    if buf:
        last_line = buf.decode(errors="replace").strip()
    inner.close()

    rc = process.wait()
    if rc != 0:
        raise RuntimeError(last_line or f"megadl exited with code {rc}")

    downloaded_files = [f for f in os.listdir(TEMP_DIR) if os.path.isfile(os.path.join(TEMP_DIR, f))]
    if not downloaded_files:
        raise RuntimeError(f"megadl exited 0 but no file appeared in temp dir. Last line: {last_line}")
    return os.path.join(TEMP_DIR, downloaded_files[0])

def move_to_drive(local_path):
    fname = os.path.basename(local_path)
    dest = os.path.join(DRIVE_DEST_PATH, fname)
    shutil.move(local_path, dest)
    return dest, fname, os.path.getsize(dest)

stopped_for_quota = False
failed_links = []

pbar = tqdm(MEGA_FILE_LINKS, desc="Transferring")
for url in pbar:
    key = link_id(url)
    pbar.set_postfix_str(key)

    # 1) Fast check: state.json
    if already_done(url, drive_files, state):
        print(f"[{key}] already in Drive (state match), skipping.")
        continue

    # 2) Real check: MEGA metadata (name+size) vs Drive — no content download yet
    is_present, fname, size = already_done_by_metadata(url, drive_files)
    if is_present:
        print(f"[{key}] '{fname}' ({size} bytes) already in Drive (metadata match), skipping download.")
        state[key] = {"filename": fname, "size": size, "status": "completed"}
        save_state(state)
        continue
    elif fname:
        print(f"[{key}] remote file: '{fname}' ({size} bytes) — not in Drive, downloading...")

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"[{key}] attempt {attempt}/{MAX_RETRIES}...")
            local_path = download_one(url, total_size=size, pbar=pbar)
            dest, dfname, dsize = move_to_drive(local_path)
            print(f"  -> saved as '{dfname}' ({dsize} bytes) to {dest}")
            state[key] = {"filename": dfname, "size": dsize, "status": "completed"}
            save_state(state)
            drive_files[dfname] = dsize
            success = True
            break
        except RuntimeError as e:
            msg = str(e)
            if is_quota_error(msg):
                print("\n🛑 MEGA bandwidth quota exceeded.")
                print("   Stopping gracefully. Re-run this cell later — completed")
                print("   files are already tracked and will be skipped.")
                stopped_for_quota = True
                break
            wait = BACKOFF_BASE_SECONDS * (2 ** (attempt - 1))
            print(f"  ERROR: {msg[:300]}")
            if attempt < MAX_RETRIES:
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
    if stopped_for_quota:
        break
    if not success:
        print(f"  ✗ Giving up on {key} after {MAX_RETRIES} attempts.")
        state[key] = {"filename": None, "size": None, "status": "failed"}
        save_state(state)
        failed_links.append(url)

print("\n=== STOPPED DUE TO QUOTA — re-run this cell later ===" if stopped_for_quota
      else "\n=== Transfer loop finished ===")
if failed_links:
    print(f"Links that failed after {MAX_RETRIES} retries: {failed_links}")

**Cell 7: Verification Report**

In [ ]:
final_drive_files = scan_drive_folder()
expected_count = len(MEGA_FILE_LINKS)
completed = [k for k, v in state.items() if v.get("status") == "completed"]
failed = [k for k, v in state.items() if v.get("status") == "failed"]

print("========== VERIFICATION REPORT ==========")
print(f"Total MEGA links given     : {expected_count}")
print(f"Marked completed in state  : {len(completed)}")
print(f"Marked failed in state     : {len(failed)}")
print(f"Files actually in Drive    : {len(final_drive_files)}")

missing = []
for k, v in state.items():
    if v.get("status") == "completed":
        fname, size = v.get("filename"), v.get("size")
        if fname not in final_drive_files or final_drive_files[fname] != size:
            missing.append((k, fname))

if missing:
    print(f"\n⚠️ {len(missing)} file(s) marked completed but NOT verified in Drive:")
    for k, fname in missing:
        print(f"   ✗ {k} -> {fname}")
elif len(completed) == expected_count:
    print("\n✅ All files successfully transferred and verified in Drive.")
else:
    print(f"\n⚠️ {expected_count - len(completed)} link(s) not completed yet — re-run Cell 6.")
print("===========================================")

**Cell 8: Notes**

## Notes
- **Resume:** Cells 2 → 6 phir se run karo, kuch duplicate nahi hoga.
- **Quota block:** Kuch ghante wait karke sirf Cell 6 phir se run karo.
- **No permanent local storage:** Files `TEMP_DIR` me temporarily aati hain, turant Drive me move ho jaati hain.